In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tabpfn import TabPFNClassifier
from tabpfn_extensions.embedding import TabPFNEmbedding

In [27]:

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [28]:
class StudentModel(nn.Module):
    def __init__(self, in_dim, emb_dim, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, emb_dim),
        )
        self.decoder = nn.Linear(emb_dim, n_classes)

    def forward(self, x):
        z = self.encoder(x)
        logits = self.decoder(z)
        return z, logits

In [29]:
teacher = TabPFNClassifier(n_estimators=1, random_state=42)
teacher.fit(X_train, y_train)

embedder = TabPFNEmbedding(model=teacher, n_fold=0)

emb_train = embedder.fit_transform(X_train, y_train)
emb_test = embedder.transform(X_test)

if emb_train.ndim == 3:
    emb_train = emb_train[0]
    emb_test = emb_test[0]

teacher_proba_train = teacher.predict_proba(X_train)
teacher_proba_test = teacher.predict_proba(X_test)

x_scaler = StandardScaler().fit(X_train)
emb_scaler = StandardScaler().fit(emb_train)

X_train_s = x_scaler.transform(X_train)
X_test_s = x_scaler.transform(X_test)

emb_train_s = emb_scaler.transform(emb_train)
emb_test_s = emb_scaler.transform(emb_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

emb_train_t = torch.tensor(emb_train_s, dtype=torch.float32)
emb_test_t = torch.tensor(emb_test_s, dtype=torch.float32)

teacher_train_t = torch.tensor(teacher_proba_train, dtype=torch.float32)

n_features = X_train.shape[1]
emb_dim = emb_train.shape[1]
n_classes = teacher_proba_train.shape[1]


student = StudentModel(n_features, emb_dim, n_classes)
opt = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)

/usr/local/lib/python3.12/dist-packages/tabpfn/validation.py:142: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
/usr/local/lib/python3.12/dist-packages/tabpfn/validation.py:142: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


epoch= 200  loss=0.153539  emb=0.153162  kl=0.000377
epoch= 400  loss=0.066479  emb=0.066288  kl=0.000192
epoch= 600  loss=0.039764  emb=0.039638  kl=0.000126
epoch= 800  loss=0.028298  emb=0.028199  kl=0.000099
epoch=1000  loss=0.021622  emb=0.021539  kl=0.000083
epoch=1200  loss=0.017616  emb=0.017546  kl=0.000070
epoch=1400  loss=0.014667  emb=0.014598  kl=0.000069
epoch=1600  loss=0.012639  emb=0.012575  kl=0.000064
epoch=1800  loss=0.010688  emb=0.010632  kl=0.000056
epoch=2000  loss=0.010267  emb=0.010212  kl=0.000055


In [ ]:
alpha = 1.0
beta = 1.0

for epoch in range(2000):
    opt.zero_grad()

    z_hat, logits_hat = student(X_train_t)
    log_probs_hat = F.log_softmax(logits_hat, dim=1)

    loss_emb = F.mse_loss(z_hat, emb_train_t)
    loss_kl = F.kl_div(log_probs_hat, teacher_train_t, reduction="batchmean")

    loss = alpha * loss_emb + beta * loss_kl
    loss.backward()
    opt.step()

    if (epoch + 1) % 200 == 0:
        print(
            f"epoch={epoch+1:4d}  "
            f"loss={loss.item():.6f}  "
            f"emb={loss_emb.item():.6f}  "
            f"kl={loss_kl.item():.6f}"
        )


In [ ]:
with torch.no_grad():
    _, logits_test = student(X_test_t)
    student_proba_test = F.softmax(logits_test, dim=1).cpu().numpy()

teacher_pred = teacher_proba_test.argmax(axis=1)
student_pred = student_proba_test.argmax(axis=1)

teacher_acc = (teacher_pred == y_test).mean()
student_acc = (student_pred == y_test).mean()
agreement = (teacher_pred == student_pred).mean()

kl = np.mean(
    np.sum(
        teacher_proba_test
        * (
            np.log(teacher_proba_test + 1e-9)
            - np.log(student_proba_test + 1e-9)
        ),
        axis=1,
    )
)

In [30]:

print("\nRESULTS")
print(f"Teacher acc       : {teacher_acc:.4f}")
print(f"Student acc       : {student_acc:.4f}")
print(f"Teacher agreement : {agreement:.4f}")
print(f"KL(T || S)        : {kl:.6f}")


RESULTS
Teacher acc       : 0.9825
Student acc       : 0.9737
Teacher agreement : 0.9737
KL(T || S)        : 0.078646
